In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Q.1

In [2]:
import pandas as pd

DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

mcq_df = pd.read_csv(DATA_PATH)

print(mcq_df.head())

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

# Q.2

In [3]:
answer_to_id = dict(zip("ABCDE", range(5)))

mcq_df = mcq_df.assign(
    target=mcq_df["answer"].replace(answer_to_id)
)

label_value = int(mcq_df.at[150, "target"])

print(f"Encoded label at row 150 : {label_value}")

Encoded label at row 150 : 2


/tmp/ipykernel_58/1676217974.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  target=mcq_df["answer"].replace(answer_to_id)


# Q.3

In [4]:
question = str(mcq_df.at[0, "prompt"])
choice = str(mcq_df.at[0, "B"])

pair_text = " [SEP] ".join([question, choice])

print(
    f"Formatted sequence length = {len(pair_text)} characters"
)

Formatted sequence length = 407 characters


# Q.4

In [5]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

question = mcq_df.at[0, "prompt"]

candidate_inputs = [
    f"{question} [SEP] {mcq_df.at[0, option]}"
    for option in ["A", "B", "C", "D", "E"]
]

encoded_batch = bert_tokenizer(
    candidate_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

reshaped_ids = encoded_batch["input_ids"].view(1, 5, 128)

print(
    f"Second tensor dimension = {reshaped_ids.size(1)}"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Second tensor dimension = 5


# Q.5

In [6]:
import torch

NUM_EXAMPLES = 16
OPTIONS = ["A", "B", "C", "D", "E"]

pairs = []

for sample in mcq_df.iloc[:NUM_EXAMPLES].itertuples():

    pairs.extend(
        f"{sample.prompt} [SEP] {getattr(sample, option)}"
        for option in OPTIONS
    )

batch_encoding = bert_tokenizer(
    pairs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

tensor = batch_encoding["input_ids"].reshape(
    NUM_EXAMPLES,
    len(OPTIONS),
    128
)

positions = tensor.numel()

print(
    f"Total token positions = {positions}"
)

Total token positions = 10240


# Q.6

In [7]:
from transformers import AutoModelForMultipleChoice
import torch

mcq_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

question = mcq_df.at[0, "prompt"]

formatted = [
    f"{question} [SEP] {mcq_df.at[0, option]}"
    for option in ["A", "B", "C", "D", "E"]
]

encoded = bert_tokenizer(
    formatted,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

model_inputs = {
    "input_ids": encoded["input_ids"].unsqueeze(0),
    "attention_mask": encoded["attention_mask"].unsqueeze(0),
}

with torch.inference_mode():
    prediction = mcq_model(**model_inputs)

logit_shape = tuple(prediction.logits.shape)

print(f"Logits tensor shape : {logit_shape}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits tensor shape : (1, 5)


# Q.7

In [8]:
# Install dependencies
!pip -q install peft
!pip -q install -U torchao>=0.16.0

from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

base_mcq = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

adapter_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"]
)

lora_model = get_peft_model(base_mcq, adapter_cfg)

trainable = sum(
    parameter.numel()
    for parameter in lora_model.parameters()
    if parameter.requires_grad
)

print(f"Trainable Parameters : {trainable}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Trainable Parameters : 295681


# Q.8

In [9]:
from datasets import Dataset

# Create a small subset for demonstration
subset = Dataset.from_pandas(mcq_df.iloc[:100].copy())

OPTION_COLUMNS = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):

    first_sentences = []
    second_sentences = []

    # Build (question, option) pairs
    for i in range(len(examples["prompt"])):
        question = examples["prompt"][i]

        for opt in OPTION_COLUMNS:
            first_sentences.append(question)
            second_sentences.append(examples[opt][i])

    # Tokenize all pairs together
    tokenized = bert_tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    batch_size = len(examples["prompt"])
    num_choices = len(OPTION_COLUMNS)

    processed = {}

    # Reshape EVERY tokenizer output
    for key, value in tokenized.items():
        processed[key] = [
            value[i * num_choices:(i + 1) * num_choices]
            for i in range(batch_size)
        ]

    processed["labels"] = examples["target"]

    return processed


tokenized_dataset = subset.map(
    preprocess_function,
    batched=True,
    remove_columns=subset.column_names,
)

print(tokenized_dataset)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


# Q.9

In [10]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForMultipleChoice
)

training_cfg = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=lora_model,
    args=training_cfg,
    train_dataset=tokenized_dataset.select(range(32)),
    data_collator=DataCollatorForMultipleChoice(
        tokenizer=bert_tokenizer
    ),
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss


TrainOutput(global_step=4, training_loss=3.321335792541504, metrics={'train_runtime': 3.041, 'train_samples_per_second': 10.523, 'train_steps_per_second': 1.315, 'total_flos': 10560681000960.0, 'train_loss': 3.321335792541504, 'epoch': 1.0})

# Q.10

In [12]:
import torch

sample = mcq_df.iloc[0]

question = sample["prompt"]

choices = [
    sample["A"],
    sample["B"],
    sample["C"],
    sample["D"],
    sample["E"],
]

encoded = bert_tokenizer(
    [question] * len(choices),
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

device = next(lora_model.parameters()).device

inputs = {
    key: value.unsqueeze(0).to(device)
    for key, value in encoded.items()
}

lora_model.eval()

with torch.no_grad():
    outputs = lora_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)

print("Option Probabilities")

for option, prob in zip(["A", "B", "C", "D", "E"], probabilities[0]):
    print(f"{option}: {prob.item():.4f}")

Option Probabilities
A: 0.1993
B: 0.2002
C: 0.2007
D: 0.2002
E: 0.1996
